# Query Expansion - Continuacion del RAG naive

Este notebook continua a `naive_rag_lab.ipynb`. Alli vimos el pipeline RAG basico: **embedding de la pregunta -> busqueda en Qdrant -> prompt -> LLM**.

## El problema

En el RAG naive generamos **un solo embedding** para la pregunta del usuario y buscamos con ese unico vector. El problema es que la forma en que un usuario redacta su pregunta no siempre coincide (lexicamente o semanticamente) con la forma en que esta redactado el contenido en los documentos indexados. Por ejemplo:

- Pregunta del usuario: *"que programas tiene?"*
- Documento indexado: *"Nuestra oferta academica incluye las siguientes carreras..."*

Aunque el significado es el mismo, el embedding de la pregunta puede no acercarse lo suficiente al embedding del documento relevante, y terminamos recuperando menos contexto util del que existe.

## La solucion: Query Expansion (multi-query fan-out)

La idea es simple: en lugar de buscar con una sola version de la pregunta, le pedimos al LLM que genere **N reformulaciones alternativas** de la misma pregunta (mismo significado, distintas palabras/estructura). Luego:

1. Buscamos en Qdrant con el embedding de **cada** variante (incluida la original).
2. **Fusionamos** todos los resultados, eliminando duplicados (nos quedamos con el mejor `score` cuando un mismo documento aparece en mas de una busqueda).
3. Nos quedamos con el **top-k** final para armar el contexto que se le pasa al LLM.

Esto aumenta la probabilidad de recuperar documentos relevantes que la pregunta original, por si sola, no hubiera encontrado.

Esto es equivalente a lo que implementamos en el backend en:
- `src/integrations/query_expansion_integration.py` (interfaz)
- `src/integrations/impl/openai_query_expansion_integration.py` (implementacion con LLM)
- `src/services/agent_service.py` (metodo `_retrieve_with_query_expansion`)

## 0. Setup

Reutilizamos exactamente la misma configuracion que en `naive_rag_lab.ipynb`: variables de entorno, cliente de Qdrant, embeddings y LLM.

In [ ]:
import os
from dotenv import load_dotenv

# Cargamos el .env de 01_rag_api (un nivel arriba de notebooks/)
load_dotenv(os.path.join("..", ".env"))

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
OPENAI_EMBEDDING_MODEL = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")
ASSISTANT_NAME = os.getenv("ASSISTANT_NAME", "Asistente")

QDRANT_URL = os.getenv("QDRANT_URL")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")
QDRANT_COLLECTION_NAME = os.getenv("QDRANT_COLLECTION_NAME")

RETRIEVAL_LIMIT = 5
QUERY_EXPANSION_COUNT = 3

print("Modelo LLM:", OPENAI_MODEL)
print("Modelo de embeddings:", OPENAI_EMBEDDING_MODEL)
print("Coleccion de Qdrant:", QDRANT_COLLECTION_NAME)

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient

# Cliente de Qdrant apuntando a la coleccion ya existente
client = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
    timeout=60,
)

# Modelo de embeddings de OpenAI (el mismo usado al indexar los documentos)
embeddings = OpenAIEmbeddings(api_key=OPENAI_API_KEY, model=OPENAI_EMBEDDING_MODEL)

# VectorStore de LangChain: envuelve el cliente + la coleccion + el modelo de embeddings
vector_store = QdrantVectorStore(
    client=client,
    collection_name=QDRANT_COLLECTION_NAME,
    embedding=embeddings,
)

# LLM que usaremos tanto para expandir la pregunta como para responder
llm = ChatOpenAI(api_key=OPENAI_API_KEY, model=OPENAI_MODEL)

info = client.get_collection(QDRANT_COLLECTION_NAME)
print("Conectado a la coleccion:", QDRANT_COLLECTION_NAME)
print("Puntos indexados:", info.points_count)

In [ ]:
question = "que programas tiene?"

print("Pregunta original:", question)

## 1. Paso 1 - Generar variantes de la pregunta con el LLM

Le pedimos al LLM que reformule la pregunta original en `N` variantes distintas, preservando el significado. Usamos `with_structured_output` de LangChain con un modelo Pydantic para que el LLM devuelva directamente una lista de strings (sin tener que parsear texto libre).

Esto es equivalente a `OpenAIQueryExpansionIntegration.expand` en `src/integrations/impl/openai_query_expansion_integration.py`, que usa las plantillas definidas en `src/integrations/impl/prompt.py` (`QUERY_EXPANSION_INSTRUCTIONS` y `QUERY_EXPANSION_USER_PROMPT`).

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

# Mismas plantillas que src/integrations/impl/prompt.py
QUERY_EXPANSION_INSTRUCTIONS = (
    "Eres un asistente experto en sistemas de recuperacion de informacion (RAG). "
    "Tu tarea es reformular la pregunta del usuario en variantes alternativas que "
    "preserven exactamente el mismo significado e intencion, pero que usen palabras, "
    "sinonimos o estructuras gramaticales distintas. Esto ayuda a recuperar mas "
    "documentos relevantes de una base vectorial. No respondas la pregunta, no agregues "
    "informacion nueva y responde siempre en el mismo idioma que la pregunta original."
)

QUERY_EXPANSION_USER_PROMPT = (
    "Genera {n} reformulaciones distintas de la siguiente pregunta:\n\n"
    "Pregunta original: {question}"
)


class QueryVariants(BaseModel):
    variants: list[str] = Field(description="Lista de reformulaciones de la pregunta original")


expansion_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", QUERY_EXPANSION_INSTRUCTIONS),
        ("human", QUERY_EXPANSION_USER_PROMPT),
    ]
)

structured_llm = llm.with_structured_output(QueryVariants)
expansion_chain = expansion_prompt | structured_llm

result = expansion_chain.invoke({"question": question, "n": QUERY_EXPANSION_COUNT})
variants = result.variants[:QUERY_EXPANSION_COUNT]

print("Pregunta original:", question)
print("\nVariantes generadas por el LLM:")
for i, v in enumerate(variants, start=1):
    print(f"  {i}. {v}")

## 2. Paso 2 - Retrieval por cada variante

Primero repetimos el retrieval naive (una sola busqueda) para tener un punto de comparacion. Luego hacemos el retrieval expandido: generamos el embedding de **cada** variante (incluyendo la pregunta original) y buscamos en Qdrant con cada uno.

In [ ]:
# Retrieval naive: una sola busqueda con la pregunta original
naive_vector = embeddings.embed_query(question)
naive_docs_with_scores = vector_store.similarity_search_with_score_by_vector(
    naive_vector, k=RETRIEVAL_LIMIT
)

print(f"Retrieval naive: {len(naive_docs_with_scores)} documentos\n")
for doc, score in naive_docs_with_scores:
    print(f"score={score:.4f} | source={doc.metadata.get('source_file', '')}")
    print(doc.page_content[:150], "...\n")

In [ ]:
# Retrieval expandido: una busqueda por cada variante (+ la pregunta original)
queries = [question] + variants

results_by_query = {}
for q in queries:
    vector = embeddings.embed_query(q)
    docs_with_scores = vector_store.similarity_search_with_score_by_vector(vector, k=RETRIEVAL_LIMIT)
    results_by_query[q] = docs_with_scores

for q, docs_with_scores in results_by_query.items():
    print(f"=== Query: {q} ===")
    for doc, score in docs_with_scores:
        print(f"  score={score:.4f} | source={doc.metadata.get('source_file', '')}")
    print()

## 3. Paso 3 - Fusion y deduplicacion de resultados

Ahora combinamos los resultados de todas las busquedas. Como el mismo documento puede aparecer en varias variantes (con distinto score), nos quedamos con el **mejor score** por documento y despues recortamos al `top-k` final (`RETRIEVAL_LIMIT`).

Esto es lo que hace `AgentService._retrieve_with_query_expansion` en el backend, usando el `id` de cada resultado (`ElementoBusqueda.id`) como clave de deduplicacion. Aqui usamos el contenido del documento como clave, ya que es lo que tenemos disponible en este notebook.

In [ ]:
# Fusionamos todos los resultados de todas las queries en un solo dict,
# quedandonos con el mejor score cuando un documento se repite.
best_by_content = {}
for docs_with_scores in results_by_query.values():
    for doc, score in docs_with_scores:
        key = doc.page_content
        if key not in best_by_content or score > best_by_content[key][1]:
            best_by_content[key] = (doc, score)

merged_docs_with_scores = sorted(best_by_content.values(), key=lambda item: item[1], reverse=True)
merged_docs_with_scores = merged_docs_with_scores[:RETRIEVAL_LIMIT]

print(f"Documentos unicos encontrados (todas las variantes): {len(best_by_content)}")
print(f"Documentos naive (1 sola query): {len(naive_docs_with_scores)}")
print(f"Top-{RETRIEVAL_LIMIT} final tras fusion:\n")
for doc, score in merged_docs_with_scores:
    print(f"score={score:.4f} | source={doc.metadata.get('source_file', '')}")
    print(doc.page_content[:150], "...\n")

## 4. Paso 4 - Construir el prompt e invocar al LLM

Igual que en el notebook naive: formateamos el contexto (ahora fusionado) y armamos el prompt de 3 partes (system + historial + human), luego lo enviamos al LLM.

In [ ]:
formatted_context = "\n".join([f"- {doc.page_content}" for doc, _score in merged_docs_with_scores])

INSTRUCTIONS = (
    "Eres un asistente útil y amigable. Responde siempre en español."
    "Tu nombre es {assistant_name}."
    "Sé conciso en tus respuestas."
)

USER_PROMPT = (
    "Contexto:\n"
    "{context}\n\n"
    "Pregunta: {question}"
)

answer_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", INSTRUCTIONS),
        ("human", USER_PROMPT),
    ]
)

answer_chain = answer_prompt | llm

response = answer_chain.invoke(
    {
        "assistant_name": ASSISTANT_NAME,
        "question": question,
        "context": formatted_context,
    }
)

print("Pregunta:", question)
print("\nRespuesta (con query expansion):\n")
print(response.content)

## 5. Funcion reusable: `rag_with_query_expansion`

Encapsulamos todo el pipeline (expandir -> buscar por variante -> fusionar -> prompt -> LLM) en una sola funcion, analoga a `naive_rag()` del notebook anterior.

In [ ]:
def rag_with_query_expansion(pregunta: str, n_variantes: int = QUERY_EXPANSION_COUNT, k: int = RETRIEVAL_LIMIT) -> str:
    # Paso 1: generar variantes de la pregunta
    result = expansion_chain.invoke({"question": pregunta, "n": n_variantes})
    variantes = result.variants[:n_variantes]
    queries = [pregunta] + variantes

    # Paso 2: retrieval por cada variante
    best_by_content = {}
    for q in queries:
        vector = embeddings.embed_query(q)
        for doc, score in vector_store.similarity_search_with_score_by_vector(vector, k=k):
            key = doc.page_content
            if key not in best_by_content or score > best_by_content[key][1]:
                best_by_content[key] = (doc, score)

    # Paso 3: fusionar y recortar al top-k
    merged = sorted(best_by_content.values(), key=lambda item: item[1], reverse=True)[:k]
    contexto = "\n".join([f"- {doc.page_content}" for doc, _score in merged])

    # Paso 4: prompt + LLM
    respuesta = answer_chain.invoke(
        {
            "assistant_name": ASSISTANT_NAME,
            "question": pregunta,
            "context": contexto,
        }
    )

    return respuesta.content


print(rag_with_query_expansion("¿Cuales son los requisitos de inscripcion?"))

## 6. Comparacion final: naive vs query expansion

Corremos la misma pregunta con el pipeline naive (`naive_rag`, definido igual que en el notebook anterior) y con `rag_with_query_expansion`, para notar la diferencia en el contexto recuperado y en la respuesta final.

In [ ]:
def naive_rag(pregunta: str, limit: int = RETRIEVAL_LIMIT) -> str:
    vector = embeddings.embed_query(pregunta)
    docs_with_scores = vector_store.similarity_search_with_score_by_vector(vector, k=limit)
    contexto = "\n".join([f"- {doc.page_content}" for doc, _score in docs_with_scores])
    respuesta = answer_chain.invoke(
        {
            "assistant_name": ASSISTANT_NAME,
            "question": pregunta,
            "context": contexto,
        }
    )
    return respuesta.content


pregunta_comparacion = "que programas tiene?"

print("=== RAG naive ===")
print(naive_rag(pregunta_comparacion))

print("\n=== RAG con query expansion ===")
print(rag_with_query_expansion(pregunta_comparacion))